# NCKH – phát hiện người bằng RetinaNet trên Kaggle

Notebook này là **bộ xử lý AI chạy khi bạn bấm Run All**, không phải API/web backend luôn trực tuyến. Backend Render miễn phí vẫn nhận video và phục vụ giao diện Next.js.

Trước khi chạy: bật **Internet** trong Settings, chọn **GPU** nếu có, vào **Add-ons → Secrets** tạo và gắn secret `SUPABASE_SECRET_KEY`. Không dán khóa vào ô mã. Chỉ xử lý video bạn có quyền sử dụng.

Luồng: web tải video → Supabase `raw-videos` và bảng `videos` → Notebook lấy video trạng thái `pending` → RetinaNet → JSON riêng tư `raw-detections` và tóm tắt trong `videos.metadata.detection`. Notebook không làm tracking/Re-ID.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path('/kaggle/working/NCKH')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', 'https://github.com/Tiens0710/NCKH.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo / 'backend' / 'requirements-kaggle.txt')], check=True)
import cv2, torch, torchvision
from PIL import Image
print('Mã sẵn sàng; thiết bị:', 'CUDA' if torch.cuda.is_available() else 'CPU')

In [ ]:
from kaggle_secrets import UserSecretsClient

secret = UserSecretsClient().get_secret('SUPABASE_SECRET_KEY')
if not secret or not secret.startswith('sb_secret_'):
    raise RuntimeError('Hãy gắn SUPABASE_SECRET_KEY trong Add-ons → Secrets; không ghi khóa vào notebook.')
os.environ['SUPABASE_URL'] = 'https://wtwjsisqghrqtqhzepci.supabase.co'
os.environ['SUPABASE_SECRET_KEY'] = secret
del secret
print('Đã nạp khóa từ Kaggle Secrets (không hiển thị giá trị).')

In [ ]:
from supabase import create_client

client = create_client(os.environ['SUPABASE_URL'], os.environ['SUPABASE_SECRET_KEY'])
pending = client.table('videos').select('id').eq('status', 'pending').execute().data
print('Số video đang chờ:', len(pending))
if not pending:
    print('Bạn có thể tải video mới trên trang Video của ứng dụng rồi chạy lại notebook.')

In [ ]:
# Xử lý tối đa 20 video chờ. 'auto' dùng GPU nếu Kaggle cấp GPU, nếu không dùng CPU.
subprocess.run(
    [sys.executable, '-m', 'backend.ai.worker', '--drain', '--max-jobs', '20', '--device', 'auto'],
    cwd=repo, env=os.environ.copy(), check=True
)

In [ ]:
rows = client.table('videos').select('status,metadata').order('created_at', desc=True).limit(20).execute().data
for row in rows:
    detection = (row.get('metadata') or {}).get('detection') or {}
    print(row['status'], '|', detection.get('device', '—'), '|', detection.get('person_boxes', 0), 'khung bao')
print('Mở lại trang Video trên web và tải lại để xem tóm tắt.')